<a href="https://colab.research.google.com/github/Murcha1990/ML_AI25/blob/main/Lesson13_Optuna%26Clustering/Optuna_screencast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Общий алгоритм работы с Optuna

In [8]:
!pip install optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 8.0 MB/s eta 0:00:00


1. Определяем целевую функцию objective, через аргументы она будет получать специальный объект trial. С его помощью можно назначать различные гипермараметры, Например, как в примере ниже, мы задаем x в интервале [-10,10].

2. Далее создаем объект обучения с помощью метода optuna.create_study.

3. Запускаем оптимизацию целевой функции objective на 10 итераций n_trials=10. Происходит 10 вызовов нашей функции с различными параметрами от -10 до 10. Какие именно параметры выбирает optuna будет описано ниже.

In [9]:
import optuna

def objective(trial):
    x = trial.suggest_float('x', -10, 10)
    return (x - 2) ** 2

study = optuna.create_study()
study.optimize(objective, n_trials=40)

study.best_params

[I 2026-02-03 14:39:46,811] A new study created in memory with name: no-name-fe3ea8bf-8e4d-4e9b-8460-093b2b7c922c
[I 2026-02-03 14:39:46,814] Trial 0 finished with value: 22.767341175870456 and parameters: {'x': -2.7715135099746346}. Best is trial 0 with value: 22.767341175870456.
[I 2026-02-03 14:39:46,816] Trial 1 finished with value: 35.098708004846 and parameters: {'x': 7.92441625857316}. Best is trial 0 with value: 22.767341175870456.
[I 2026-02-03 14:39:46,817] Trial 2 finished with value: 46.8541821145595 and parameters: {'x': 8.845011476583476}. Best is trial 0 with value: 22.767341175870456.
[I 2026-02-03 14:39:46,819] Trial 3 finished with value: 17.09403865948263 and parameters: {'x': -2.134493760967917}. Best is trial 3 with value: 17.09403865948263.
[I 2026-02-03 14:39:46,822] Trial 4 finished with value: 75.52209331703958 and parameters: {'x': -6.690344833033932}. Best is trial 3 with value: 17.09403865948263.
[I 2026-02-03 14:39:46,824] Trial 5 finished with value: 63.43

{'x': 1.9449468950446716}

## Загрузка данных и импорт библиотек

In [1]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score

from sklearn.datasets import fetch_california_housing

In [2]:
RANDOM_STATE = 42

In [3]:
!pip install lightgbm -q

In [4]:
from lightgbm import LGBMRegressor

In [5]:
data = fetch_california_housing(as_frame=True)

X = data.data
y = data.target

In [6]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)

## Подбор гиперпараметров с Optuna

Разобъем данные на тренировочную и тестовую часть. На тренировочной части по кросс-валидации подберем гиперпараметры моделей, а затем проверим качество на тестовой части.

In [10]:
def objective_lgbm(trial):
    max_depth = trial.suggest_int("max_depth", 2, 20)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1, log=True)
    n_estimators = trial.suggest_int("n_estimators", 10, 1000)

    score = cross_val_score(LGBMRegressor(max_depth=max_depth, learning_rate=learning_rate, n_estimators=n_estimators),
                            Xtrain, ytrain, cv=3, scoring='r2', n_jobs=-1).mean()
    return score


study = optuna.create_study(direction="maximize")
study.optimize(objective_lgbm, n_trials=30)

[I 2026-02-03 14:43:20,541] A new study created in memory with name: no-name-57bce5b6-d50b-4761-9cbc-b6704dc4d4a5
[I 2026-02-03 14:43:26,744] Trial 0 finished with value: 0.028749278629490987 and parameters: {'max_depth': 2, 'learning_rate': 4.6610739654569813e-05, 'n_estimators': 717}. Best is trial 0 with value: 0.028749278629490987.
[I 2026-02-03 14:43:32,141] Trial 1 finished with value: 0.8400615613624368 and parameters: {'max_depth': 10, 'learning_rate': 0.022583994236058558, 'n_estimators': 991}. Best is trial 1 with value: 0.8400615613624368.
[I 2026-02-03 14:43:38,927] Trial 2 finished with value: 0.795310654404323 and parameters: {'max_depth': 14, 'learning_rate': 0.004167870562801832, 'n_estimators': 768}. Best is trial 1 with value: 0.8400615613624368.
[I 2026-02-03 14:43:40,159] Trial 3 finished with value: 0.8388678742863606 and parameters: {'max_depth': 14, 'learning_rate': 0.12393254786279155, 'n_estimators': 269}. Best is trial 1 with value: 0.8400615613624368.
[I 2026

In [11]:
study.best_params

{'max_depth': 10, 'learning_rate': 0.022583994236058558, 'n_estimators': 991}

In [12]:
model = LGBMRegressor(**study.best_params)
model.fit(Xtrain, ytrain)

pred = model.predict(Xtest)

r2_score(ytest, pred)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001342 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1838
[LightGBM] [Info] Number of data points in the train set: 15480, number of used features: 8
[LightGBM] [Info] Start training from score 2.070349


0.8521645440059706

В Optuna встроена гибкая возможность перебора гиперпараметров.

In [ ]:
# 1. Define an objective function to be maximized.
def objective(trial):

    # 2. Suggest values for the hyperparameters using a trial object.
    classifier_name = trial.suggest_categorical('classifier', ['SVC', 'RandomForest'])
    if classifier_name == 'SVC':
         svc_c = trial.suggest_float('svc_c', 1e-10, 1e10, log=True)
         classifier_obj = sklearn.svm.SVC(C=svc_c, gamma='auto')
    else:
        rf_max_depth = trial.suggest_int('rf_max_depth', 2, 32, log=True)
        classifier_obj = sklearn.ensemble.RandomForestClassifier(max_depth=rf_max_depth, n_estimators=10)
    ...
    return accuracy

# 3. Create a study object and optimize the objective function.
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)